# Beer Peer
## Advanced Machine Learning — Final Project

---

### Product Description

Beer Peer is a beer recommendation app. Photograph your food — that is the only input. The app returns three beer style recommendations with flavor descriptions written by real beer drinkers.

The pipeline is three independent layers:

```
PHOTO  →  CNN  →  food label  →  lookup table  →  beer styles  →  BiLSTM  →  flavor language
```

---

### Example Output

| | |
|---|---|
| **Input** | Photo of pizza margherita |
| **CNN output** | Pizza (94% confidence) |
| **Paired styles** | IPA · APA · Lager |
| **IPA** | "Citrusy, hoppy, bitter finish that cuts through rich tomato sauce and melted cheese." |
| **APA** | "Balanced hop character with a clean malt backbone — lifts the saltiness without overpowering." |
| **Lager** | "Crisp and clean. Light bitterness that refreshes the palate between bites." |

---

### Models

| Component | Dataset | Task |
|---|---|---|
| CNN (scratch) | Food-101 (101k images) | 101-class food classification |
| CNN (ResNet-50) | Food-101 | Same — compare vs. scratch |
| LSTM baseline | BeerAdvocate (1.5M reviews) | 8-class beer macro-style classification |
| BiLSTM + attention | BeerAdvocate | Same — compare vs. unidirectional |
| Joint Model *(+10 pts)* | Food-101 + BeerAdvocate pairs | Food-beer compatibility: 0/1 |

---

### Joint Model Design

The joint model learns food-beer compatibility from pairs supervised by the lookup table:

- **Positive pairs:** (food image, beer review) where the lookup table says they pair → label `1`
- **Negative pairs:** (food image, beer review) for a non-paired style → label `0`
- **Architecture:** frozen CNN encoder (512-d) + frozen BiLSTM encoder (256-d) → concat (768-d) → FC → sigmoid

After training, the joint model can score food-beer pairs **not in the lookup table** — generalizing the hardcoded rules into a learned compatibility function.

---

### Datasets

| Dataset | Content | Used for |
|---|---|---|
| Food-101 (`torchvision.datasets.Food101`) | 101,000 labeled food images | CNN training |
| BeerAdvocate (`rdoume/beerreviews`) | 1.5M reviews, beer_style, review_text | BiLSTM training |
| `food_beer_pairing.csv` (embedded) | 101 foods × 3 beer styles | Lookup + joint model labels |

**BiLSTM labels:** 100+ raw beer styles → 8 macro-classes (Lager / IPA / Stout-Porter / Wheat / Sour-Farmhouse / Amber-Brown / Pale Ale / Specialty)

---

### Development Phases

| Phase | Environment | Sections | Needs GPU | What happens |
|---|---|---|---|---|
| **Phase 1** | VS Code (CPU) | 1, 2, 3, 4, 6, 10, 11 | No | Environment setup. Data loading. EDA (images + text). Text preprocessing. LSTM baseline. BiLSTM with attention. |
| **Phase 2** | Google Colab (GPU) | 5, 7, 8, 13 | **Yes** | Image preprocessing + data loaders. CNN from scratch. CNN ResNet-50. Joint model full training. |
| **Phase 3** | VS Code (CPU) | 9, 12, 14, 15, deployment | No | Grad-CAM explainability. BiLSTM attention explainability. Recommendation card + 20-example table. Business framing + ethics. HF Spaces deployment. |

---

### Notebook Structure

| Section | Content |
|---|---|
| 1 | Environment setup — dependencies and imports |
| 2 | Data loading — Food-101 + BeerAdvocate + pairing table |
| 3 | EDA — image dataset (class distribution, sample grid) |
| 4 | EDA — text dataset (review length, style distribution, word clouds) |
| 5 | Image preprocessing and data loaders |
| 6 | Text preprocessing and data loaders |
| 7 | CNN — custom architecture (trained from scratch) |
| 8 | CNN — ResNet-50 (transfer learning) |
| 9 | CNN explainability — Grad-CAM |
| 10 | LSTM — unidirectional baseline |
| 11 | BiLSTM — bidirectional with attention |
| 12 | BiLSTM explainability — attention weights |
| 13 | Joint model — food-beer compatibility classifier *(+10 bonus)* |
| 14 | Business integration — recommendation card and 20-example table |
| 15 | Business framing, ethics, and team contributions |

---


## Section 1 — Environment Setup

Quick environment check — confirm Colab + CUDA before installing packages.


In [ ]:
import sys, torch

IN_COLAB = "google.colab" in sys.modules
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"{'Environment':<20} {'Google Colab' if IN_COLAB else 'Local'}")
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}")

if not IN_COLAB:
    print("\n⚠  Not running in Google Colab — GPU sections (5, 7, 8, 13) require Colab.")
elif not torch.cuda.is_available():
    print("\n⚠  CUDA not available — enable the GPU runtime in Colab: Runtime → Change runtime type → T4 GPU.")
else:
    print("\n✓ Environment check passed — Colab + CUDA confirmed.")


Environment          Local
Device               cpu
CUDA available       False


AssertionError: ⚠  Not running in Google Colab — switch to Colab for GPU training.

### 1.1 — pip installs

If you need to install all required packages, run the cell below.
It auto-detects the environment (local CPU vs. Google Colab GPU) and installs only what is missing.


In [1]:
import subprocess, sys, importlib.util

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def is_installed(import_name):
    """Check if a package is installed without importing it (avoids __init__ side-effects)."""
    return importlib.util.find_spec(import_name) is not None

# ── Detect environment ────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

# ── Install PyTorch with correct backend ──────────────────────────────────────
if is_installed("torch"):
    import torch
    print(f"  OK  torch {torch.__version__}")
else:
    print("  INSTALLING  torch ...")
    if IN_COLAB:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu118")
    else:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cpu")
    print("  DONE  torch")

# ── Install remaining packages ────────────────────────────────────────────────
# Keys = import name (used for find_spec), values = pip install name
PACKAGES = {
    "kaggle":     "kaggle",
    "nltk":       "nltk",
    "torchcam":   "torchcam",
    "wordcloud":  "wordcloud",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "pandas":     "pandas",
    "sklearn":    "scikit-learn",
    "PIL":        "Pillow",
    "ipywidgets": "ipywidgets",   # needed for tqdm notebook progress bars
}

for pkg, install_name in PACKAGES.items():
    if is_installed(pkg):
        print(f"  OK  {pkg}")
    else:
        print(f"  INSTALLING  {pkg} ...")
        pip_install(install_name)
        print(f"  DONE  {pkg}")

print("\nAll packages ready.")


Environment: Local
  OK  torch 2.11.0+cpu
  OK  kaggle
  OK  nltk
  OK  torchcam
  OK  wordcloud
  OK  matplotlib
  OK  seaborn
  OK  pandas
  OK  sklearn
  OK  PIL
  OK  ipywidgets

All packages ready.


### 1.2 — Library imports

Import all libraries, set the random seed, detect the device, and create project folders.


In [6]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import random
import string
import importlib
from pathlib import Path
from collections import Counter

# Suppress HF Hub symlinks warning on Windows (fallback to copies, still works)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud
from PIL import Image

# ── PyTorch core ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Computer vision ───────────────────────────────────────────────────────────
import torchvision.transforms as T
import torchvision.models as models
import torchvision.datasets as tv_datasets

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize

# ── Sklearn utilities ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# ── Environment ───────────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Project directories ───────────────────────────────────────────────────────
BASE_DIR = Path(".")
WEIGHTS  = BASE_DIR / "weights"
FIGURES  = BASE_DIR / "figures"
DEPLOY   = BASE_DIR / "deployment"
DATA_DIR = BASE_DIR / "data"

for d in [WEIGHTS, FIGURES, DEPLOY, DATA_DIR]:
    d.mkdir(exist_ok=True)

# ── NLTK data ─────────────────────────────────────────────────────────────────
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# ── Library versions ──────────────────────────────────────────────────────────
libs = [
    "torch", "torchvision", "numpy", "pandas",
    "matplotlib", "seaborn", "PIL", "sklearn",
    "nltk", "wordcloud",
]

print(f"{'Library':<20} {'Version'}")
print("-" * 35)
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        print(f"{lib:<20} {getattr(mod, '__version__', 'n/a')}")
    except ImportError:
        print(f"{lib:<20} NOT INSTALLED")

print()
print(f"{'Environment':<20} {'Google Colab' if IN_COLAB else 'Local'}")
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}")
print(f"{'Directories':<20} {[str(d) for d in [WEIGHTS, FIGURES, DEPLOY, DATA_DIR]]}")
print("\nSection 1 complete.")


Library              Version
-----------------------------------
torch                2.10.0+cu128
torchvision          0.25.0+cu128
numpy                2.0.2
pandas               2.2.2
matplotlib           3.10.0
seaborn              0.13.2
PIL                  11.3.0
sklearn              1.6.1
nltk                 3.9.1
wordcloud            1.9.6

Environment          Google Colab
Device               cuda
CUDA available       True
Directories          ['weights', 'figures', 'deployment', 'data']

Section 1 complete.


---

## Section 2 — Data Loading

Load all three data sources used in this project. Each subsection loads one source, validates it, and prints a summary. No preprocessing yet — raw data only.

---

### 2.1 — Load Food-101

Load the image dataset from HuggingFace Hub using `load_dataset("ethz/food101")`.

**Expected result:** DatasetDict with `train` (75,750 images) and `validation` (25,250 images) splits across 101 food classes.

**Validation checks:**
- Train + validation split sizes are correct
- Number of unique labels equals 101
- No null labels


In [3]:
# ── 2.1  Load Food-101 via torchvision ───────────────────────────────────────
# torchvision.datasets.Food101 downloads to DATA_DIR/food-101/ on first run.
# Subsequent runs load from the local cache — no network needed.
FOOD101_ROOT = DATA_DIR  # downloads into DATA_DIR/food-101/

print("Loading Food-101 (train split) …")
ds_train = tv_datasets.Food101(root=FOOD101_ROOT, split="train", download=True)
print("Loading Food-101 (test split) …")
ds_test  = tv_datasets.Food101(root=FOOD101_ROOT, split="test",  download=True)
print("Done.\n")

# ── Inspect structure ─────────────────────────────────────────────────────────
train_size  = len(ds_train)
test_size   = len(ds_test)
class_names = ds_train.classes          # list of 101 food names
n_classes   = len(class_names)

print(f"{'Split':<15} {'Rows':>8}")
print("-" * 25)
print(f"{'train':<15} {train_size:>8,}")
print(f"{'test':<15} {test_size:>8,}")
print(f"{'total':<15} {train_size + test_size:>8,}")
print()
print(f"Classes : {n_classes}")
print(f"Example labels: {class_names[:5]} … {class_names[-5:]}")

# ── Validation ────────────────────────────────────────────────────────────────
assert train_size == 75_750, f"Expected 75,750 train rows, got {train_size}"
assert test_size  == 25_250, f"Expected 25,250 test rows, got {test_size}"
assert n_classes  == 101,    f"Expected 101 classes, got {n_classes}"

# Spot-check: first 10 labels are valid integers in [0, 100]
sample_labels = [ds_train[i][1] for i in range(10)]
assert all(0 <= l < 101 for l in sample_labels), "Invalid labels found in train split"

print("\n✓ Section 2.1 validation passed — Food-101 loaded correctly.")


Loading Food-101 (train split) …


100%|██████████| 5.00G/5.00G [03:44<00:00, 22.3MB/s] 


Loading Food-101 (test split) …
Done.

Split               Rows
-------------------------
train             75,750
test              25,250
total            101,000

Classes : 101
Example labels: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare'] … ['tacos', 'takoyaki', 'tiramisu', 'tuna_tartare', 'waffles']

✓ Section 2.1 validation passed — Food-101 loaded correctly.


---

### 2.2 — Load BeerAdvocate Reviews

Download and load the BeerAdvocate CSV from Kaggle using the Kaggle Python API.
Requires `~/.kaggle/kaggle.json` to be set up once before running.

**Expected result:** DataFrame with ~1.5 million rows and columns including `beer_style`, `review_text`, `review_overall`, `review_aroma`, `review_taste`.

**Validation checks:**
- Row count > 1,000,000
- Required columns all present
- `beer_style` and `review_text` have no nulls

---

### 2.3 — Map beer styles to 8 macro-classes

Group the 100+ raw `beer_style` values into 8 meaningful macro-style classes for BiLSTM classification.

**Macro-classes:** Lager · IPA · Stout-Porter · Wheat · Sour-Farmhouse · Amber-Brown · Pale Ale · Specialty

**Expected result:** New `macro_style` column added to the DataFrame. Rows with unmapped styles are dropped.

**Validation checks:**
- Exactly 8 unique values in `macro_style`
- Class distribution is reasonably balanced (print counts)
- No nulls in `macro_style`

---

### 2.4 — Load food-beer pairing table

The pairing table is embedded directly as a CSV string — no file dependency.

**Expected result:** DataFrame with 101 rows and 4 columns: `food`, `beer_style_1`, `beer_style_2`, `beer_style_3`.

**Validation checks:**
- Exactly 101 rows
- All 4 columns present
- All beer style values in the table match one of the 8 macro-classes

---

### 2.5 — Dataset summary

Print a combined summary of all three loaded datasets before moving to EDA.
